## Abstract Factory
---

Let $F$ be a set of family keys. For each $i \in \{1, \ldots, n\}$, let $\mathcal{T}_i$ be the set of all **instances** of the $i$-th abstract product type (e.g. $\mathcal{T}_1$ = all Button instances, $\mathcal{T}_2$ = all Checkbox instances). Each $\mathcal{T}_i$ is partitioned by family:

$$\mathcal{T}_i = \bigsqcup_{k \in F} \mathcal{T}_i^{(k)}$$

where $\mathcal{T}_i^{(k)}$ is the set of instances of the concrete type for family $k$, slot $i$ (e.g. $\mathcal{T}_1^{(\text{"dark"})}$ = all DarkButton instances).

The Abstract Factory defines:

$$f : F \rightarrow \mathcal{T}_1 \times \mathcal{T}_2 \times \cdots \times \mathcal{T}_n, \qquad f(k) = \bigl(T_1^{(k)}(),\ T_2^{(k)}(),\ \ldots,\ T_n^{(k)}()\bigr)$$

where $T_i^{(k)}()$ constructs a fresh instance in $\mathcal{T}_i^{(k)}$.

$$f(\text{"dark"}) = (\text{DarkButton()},\ \text{DarkCheckbox()},\ \text{DarkMenu()})$$

$$f(\text{"light"}) = (\text{LightButton()},\ \text{LightCheckbox()},\ \text{LightMenu()})$$

**Family slice** — for each $k \in F$, define:

$$\mathcal{F}_k = \mathcal{T}_1^{(k)} \cup \mathcal{T}_2^{(k)} \cup \cdots \cup \mathcal{T}_n^{(k)}$$

**Consistency condition** — all components of $f(k)$ belong to the same family slice:

$$\forall i \in \{1, \ldots, n\}, \quad T_i^{(k)}() \in \mathcal{T}_i^{(k)} \subseteq \mathcal{F}_k$$

You can never get a `DarkButton` paired with a `LightCheckbox` — the factory enforces family purity.

**Relationship to Factory Method:**

$$\text{Factory Method}: \quad f(k) = T_k() \quad \text{(scalar output, } n = 1\text{)}$$

$$\text{Abstract Factory}: \quad f(k) = \bigl(T_1^{(k)}(),\ T_2^{(k)}(),\ \ldots,\ T_n^{(k)}()\bigr) \quad \text{(vector output)}$$

Abstract Factory strictly generalises Factory Method: setting $n = 1$ recovers it exactly.


### Exercise 1 — UI Theme Factory
---

**Scenario:** Your app supports Dark and Light themes. Each theme has its own version of a `Button` and `Checkbox`. When the user switches themes, everything must update consistently.

**Your task:** Build a `DarkThemeFactory` and `LightThemeFactory`, each producing matching `Button` and `Checkbox` objects.

```python
factory = DarkThemeFactory()
button = factory.create_button()
checkbox = factory.create_checkbox()
button.render()    # Dark Button
checkbox.render()  # Dark Checkbox
```

In [ ]:
#--------------------------------

class DarkButton:
    def render(self):
        return "Dark Button"

class DarkCheckbox:
    def render(self):
        return "Dark Checkbox"

class LightButton:
    def render(self):
        return "Light Button"

class LightCheckbox:
    def render(self):
        return "Light Checkbox"


class DarkMenu:
    def render(self):
        return "Dark Menu"

class LightMenu:
    def render(self):
        return "Light Menu"

#--------------------------------
# Note: 
# T_1=buttons = [DarkButton,LightButton]
# T_2=checkboxes = [DarkCheckbox,LightCheckbox]
# T_3=menus = [DarkMenu,LightMenu]

# Partitions:
# F={dark,light}
# T_i^dark = [DarkButton,DarkCheckbox,DarkMenu]
# T_i^light = [LightButton,LightCheckbox,LightMenu]
#--------------------------------

class DarkThemeFactory:
    F_dark={
        "button": DarkButton,
        "checkbox": DarkCheckbox,
        "menu": DarkMenu
    }
    def create_button(self):
        return DarkThemeFactory.T["button"]()

    def create_checkbox(self):
        return DarkThemeFactory.T["checkbox"]()

    def create_menu(self):
        return DarkThemeFactory.T["menu"]()

class LightThemeFactory:
    F_light={
        "button": LightButton,
        "checkbox": LightCheckbox,
        "menu": LightMenu
    }
    def create_button(self):
        return LightThemeFactory.T["button"]()

    def create_checkbox(self):
        return LightThemeFactory.T["checkbox"]()

    def create_menu(self):
        return LightThemeFactory.T["menu"]()

#--------------------------------
factory = DarkThemeFactory()
button = factory.create_button()
checkbox = factory.create_checkbox()
menu = factory.create_menu()
print(button.render())
print(checkbox.render())
print(menu.render())


Dark Button
Dark Checkbox
Dark Menu


Can we add an abstract class for the Factories?

---

In [19]:
from abc import ABC, abstractmethod

class ThemeFactory(ABC):
    @property
    @abstractmethod
    def family(self): #Forces all subclasses to implement this method
        pass
    
    def create_button(self):
        return self.family["button"]()

    def create_checkbox(self):
        return self.family["checkbox"]()

    def create_menu(self):
        return self.family["menu"]()

#--------------------------------
class DarkThemeFactory(ThemeFactory):
    F_dark={
    "button": DarkButton,
    "checkbox": DarkCheckbox,
    "menu": DarkMenu
}
    @property
    def family(self):
        return DarkThemeFactory.F_dark

class LightThemeFactory(ThemeFactory):
    F_light={
    "button": LightButton,
    "checkbox": LightCheckbox,
    "menu": LightMenu
}
    @property
    def family(self):
        return LightThemeFactory.F_light

#--------------------------------

factory = DarkThemeFactory()
button = factory.create_button()
checkbox = factory.create_checkbox()
menu = factory.create_menu()
print(button.render())
print(checkbox.render())
print(menu.render())

Dark Button
Dark Checkbox
Dark Menu



### Exercise 2 — Cross-Platform Widgets

---
**Scenario:** Your desktop app runs on both Windows and Mac. The `Toolbar` and `Menu` look different on each platform, but the app logic that uses them should be identical.

**Your task:** Create a `WindowsFactory` and `MacFactory`, each producing a `Toolbar` and `Menu`.

```python
factory = WindowsFactory()
factory.create_toolbar().render()  # Windows Toolbar
factory.create_menu().render()     # Windows Menu
```

In [ ]:
# F={windows,mac}
# T_1=[WindowsToolbar,MacToolbar]
# T_2=[WindowsMenu,MacMenu]
# So, partions are: T_i^windows = [WindowsToolbar,WindowsMenu] and T_i^mac = [MacToolbar,MacMenu]
#--------------------------------

class WindowsToolbar:
    def render(self):
        return "Windows Toolbar"

class MacToolbar:
    def render(self):
        return "Mac Toolbar"

class WindowsMenu:
    def render(self):
        return "Windows Menu"

class MacMenu:
    def render(self):
        return "Mac Menu"

#--------------------------------

class PlatformFactory(ABC):
    @property
    @abstractmethod
    def family(self):
        pass

    def create_toolbar(self):
        return self.family["toolbar"]()

    def create_menu(self):
        return self.family["menu"]()

#--------------------------------

class WindowsFactory(PlatformFactory):
    F_windows={
        "toolbar": WindowsToolbar,
        "menu": WindowsMenu
    }

    @property
    def family(self):
        return WindowsFactory.F_windows

class MacFactory(PlatformFactory):
    F_mac={
        "toolbar": MacToolbar,
        "menu": MacMenu
    }
    @property
    def family(self):
        return MacFactory.F_mac

#--------------------------------

factory = WindowsFactory()
toolbar = factory.create_toolbar()
menu = factory.create_menu()
print(toolbar.render())
print(menu.render())

    


Windows Toolbar
Windows Menu
